In [1]:
#first we exported data from MySQL Workbench using the following queries 
'''
SELECT 
    o.order_id,
    t.product_category_name_english
FROM order_items oi
JOIN orders o ON oi.order_id = o.order_id
JOIN products p ON oi.product_id = p.product_id
LEFT JOIN product_category_name t 
    ON p.product_category_name = t.product_category_name
WHERE t.product_category_name_english IS NOT NULL
ORDER BY o.order_id
INTO OUTFILE 'C:/ProgramData/MySQL/MySQL Server 8.0/Uploads/association.csv'
FIELDS TERMINATED BY ','
ENCLOSED BY '"'
LINES TERMINATED BY '\n';
'''
#We have only exported order id and category to save computation time and power

'\nSELECT \n    o.order_id,\n    t.product_category_name_english\nFROM order_items oi\nJOIN orders o ON oi.order_id = o.order_id\nJOIN products p ON oi.product_id = p.product_id\nLEFT JOIN product_category_name t \n    ON p.product_category_name = t.product_category_name\nWHERE t.product_category_name_english IS NOT NULL\nORDER BY o.order_id\nINTO OUTFILE \'C:/ProgramData/MySQL/MySQL Server 8.0/Uploads/association.csv\'\nFIELDS TERMINATED BY \',\'\nENCLOSED BY \'"\'\nLINES TERMINATED BY \'\n\';\n'

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
df = pd.read_csv('association.csv',header=None,names=['order_id', 'category'])
df

,order_id,category
0,00010242fe8c5a6d1ba2dd792cb16214,cool_stuff
1,00018f77f2f0320c557190d7a144bdd3,pet_shop
2,000229ec398224ef6ca0657da4fc703e,furniture_decor
3,00024acbcdf0a6daa1e931b038114c75,perfumery
4,00042b26cf59d7ce69dfabb4e55b4fd9,garden_tools
...,...,...
111041,fffc94f6ce00a00581880bf54a75a037,housewares
111042,fffcd46ef2263f404302a634eb57f7eb,computers_accessories
111043,fffce4705a9662cd70adb13d4a31832d,sports_leisure
111044,fffe18544ffabc95dfada21779c9644f,computers_accessories


In [4]:
#Below we are Cleaning and Handaling and exploring data

In [5]:
len(df['category'].unique())

73

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 111046 entries, 0 to 111045
Data columns (total 2 columns):
 #   Column    Non-Null Count   Dtype
---  ------    --------------   -----
 0   order_id  111046 non-null  str  
 1   category  111046 non-null  str  
dtypes: str(2)
memory usage: 6.5 MB


In [7]:
df.describe(include = 'str')

,order_id,category
count,111046,111046
unique,97276,73
top,8272b63d03f5f79c56e9e4120aec44ef,bed_bath_table
freq,21,11115


In [8]:
#Here Grooping all orders based on order id and creating the basket
#So for a particular order we have a list of items storesd in a 2D list 
basket = df.groupby('order_id')['category'].apply(list).reset_index()
basket.head(3)

,order_id,category
0,00010242fe8c5a6d1ba2dd792cb16214,[cool_stuff]
1,00018f77f2f0320c557190d7a144bdd3,[pet_shop]
2,000229ec398224ef6ca0657da4fc703e,[furniture_decor]


In [9]:
#Only saving categories of a particular order in a row 
transactions = basket['category'].tolist()

In [10]:
#To check how many orders have more than 1 unique category
basket = df.groupby('order_id')['category'].apply(list)
total = len(basket)
single_item = basket.apply(lambda x: len(set(x)) == 1).sum()
multi_item  = basket.apply(lambda x: len(set(x)) > 1).sum()

print(f"Total: {total}")
print(f"Single category orders: {single_item}")
print(f"Multi category orders:  {multi_item}")
print(f"Multi-item %: {multi_item/(single_item+multi_item)*100:.1f}%")

Total: 97276
Single category orders: 96549
Multi category orders:  727
Multi-item %: 0.7%


In [32]:
basket = df.groupby('order_id')['category'].apply(lambda x: list(set(x))) #Dont remove this even though we have this above
print(basket.head())

# Filtering multi-category orders, 
multi_basket = basket[basket.apply(lambda x: len(x) > 1)]
print(f"\n\nTotal transactions for ARM: {len(multi_basket)}")

transactions = multi_basket.tolist()

order_id
00010242fe8c5a6d1ba2dd792cb16214         [cool_stuff]
00018f77f2f0320c557190d7a144bdd3           [pet_shop]
000229ec398224ef6ca0657da4fc703e    [furniture_decor]
00024acbcdf0a6daa1e931b038114c75          [perfumery]
00042b26cf59d7ce69dfabb4e55b4fd9       [garden_tools]
Name: category, dtype: object


Total transactions for ARM: 727


In [12]:
#apriori algorithm with support = 0.01 and lift = 1
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

te = TransactionEncoder()
te_array = te.fit_transform(transactions)
basket_df = pd.DataFrame(te_array, columns=te.columns_)

frequent_itemsets_a = apriori(basket_df, min_support=0.01, use_colnames=True)
print(f"Frequent itemsets: {len(frequent_itemsets_a)}")

rules = association_rules(frequent_itemsets_a, metric="lift", min_threshold=1)
rules = rules.sort_values('lift', ascending=False)
rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(20)

Frequent itemsets: 48


,antecedents,consequents,support,confidence,lift
17,frozenset({health_beauty}),frozenset({perfumery}),0.016506,0.171429,4.793407
16,frozenset({perfumery}),frozenset({health_beauty}),0.016506,0.461538,4.793407
6,frozenset({home_confort}),frozenset({bed_bath_table}),0.059147,0.860000,3.157677
7,frozenset({bed_bath_table}),frozenset({home_confort}),0.059147,0.217172,3.157677
3,frozenset({baby}),frozenset({toys}),0.026135,0.204301,2.970538
2,frozenset({toys}),frozenset({baby}),0.026135,0.380000,2.970538
1,frozenset({baby}),frozenset({cool_stuff}),0.027510,0.215054,2.368850
0,frozenset({cool_stuff}),frozenset({baby}),0.027510,0.303030,2.368850
11,frozenset({furniture_decor}),frozenset({construction_tools_lights}),0.015131,0.054187,2.188560
10,frozenset({construction_tools_lights}),frozenset({furniture_decor}),0.015131,0.611111,2.188560


In [13]:
#apriori algorithm with support = 0.05 and lift = 1
frequent_itemsets_a_a = apriori(basket_df, min_support=0.05, use_colnames=True)
print(f"Frequent itemsets: {len(frequent_itemsets_a_a)}")
print(frequent_itemsets_a_a[frequent_itemsets_a_a['itemsets'].apply(lambda x: len(x) > 1)])

rules = association_rules(frequent_itemsets_a_a, metric="lift", min_threshold=0.5)
rules = rules.sort_values('lift', ascending=False)
rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(20)

Frequent itemsets: 14
     support                                      itemsets
12  0.096286  frozenset({bed_bath_table, furniture_decor})
13  0.059147     frozenset({home_confort, bed_bath_table})


,antecedents,consequents,support,confidence,lift
2,frozenset({home_confort}),frozenset({bed_bath_table}),0.059147,0.860000,3.157677
3,frozenset({bed_bath_table}),frozenset({home_confort}),0.059147,0.217172,3.157677
1,frozenset({furniture_decor}),frozenset({bed_bath_table}),0.096286,0.344828,1.266109
0,frozenset({bed_bath_table}),frozenset({furniture_decor}),0.096286,0.353535,1.266109


In [14]:
#Fp growth algorithm with minimum support = 0.01 and lift = 1
from mlxtend.frequent_patterns import fpgrowth, association_rules

frequent_itemsets_f = fpgrowth(basket_df, min_support=0.01, use_colnames=True)
rules = association_rules(frequent_itemsets_f, metric="lift", min_threshold=1)
rules = rules.sort_values('lift', ascending=False)
rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']]

,antecedents,consequents,support,confidence,lift
7,frozenset({health_beauty}),frozenset({perfumery}),0.016506,0.171429,4.793407
6,frozenset({perfumery}),frozenset({health_beauty}),0.016506,0.461538,4.793407
20,frozenset({home_confort}),frozenset({bed_bath_table}),0.059147,0.860000,3.157677
21,frozenset({bed_bath_table}),frozenset({home_confort}),0.059147,0.217172,3.157677
0,frozenset({toys}),frozenset({baby}),0.026135,0.380000,2.970538
1,frozenset({baby}),frozenset({toys}),0.026135,0.204301,2.970538
15,frozenset({baby}),frozenset({cool_stuff}),0.027510,0.215054,2.368850
14,frozenset({cool_stuff}),frozenset({baby}),0.027510,0.303030,2.368850
16,frozenset({construction_tools_lights}),frozenset({furniture_decor}),0.015131,0.611111,2.188560
17,frozenset({furniture_decor}),frozenset({construction_tools_lights}),0.015131,0.054187,2.188560


In [15]:
#Fp growth with minimum support = 0.05 and lift = 1
frequent_itemsets_f_f = fpgrowth(basket_df, min_support=0.05, use_colnames=True)
rules = association_rules(frequent_itemsets_f_f, metric="lift", min_threshold=1)
rules = rules.sort_values('lift', ascending=False)
rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']]

,antecedents,consequents,support,confidence,lift
2,frozenset({home_confort}),frozenset({bed_bath_table}),0.059147,0.860000,3.157677
3,frozenset({bed_bath_table}),frozenset({home_confort}),0.059147,0.217172,3.157677
1,frozenset({furniture_decor}),frozenset({bed_bath_table}),0.096286,0.344828,1.266109
0,frozenset({bed_bath_table}),frozenset({furniture_decor}),0.096286,0.353535,1.266109


In [25]:
'''
here we have tried association rules on a seasonal data but we could not find any rules.
This shows that we dont have enough data to get basket analysis using association rules
SOLUTION: For now noramal seasonal patterns can be used for further analysis.
'''

'\nhere we have tried association rules on a seasonal data but we could not find any rules.\nThis shows that we dont have enough data to get basket analysis using association rules\nSOLUTION: For now noramal seasonal patterns can be used for further analysis.\n'

In [26]:
'''
Here is the sql query for extracting useful features.
SELECT 
    o.order_id,
    t.product_category_name_english,
    o.order_purchase_timestamp,
    MONTH(o.order_purchase_timestamp) AS month,
    CASE 
        WHEN MONTH(o.order_purchase_timestamp) IN (12, 1, 2)  THEN 'Winter'
        WHEN MONTH(o.order_purchase_timestamp) IN (3, 4, 5)   THEN 'Spring'
        WHEN MONTH(o.order_purchase_timestamp) IN (6, 7, 8)   THEN 'Summer'
        WHEN MONTH(o.order_purchase_timestamp) IN (9, 10, 11) THEN 'Fall'
    END AS season
FROM order_items oi
JOIN orders o ON oi.order_id = o.order_id
JOIN products p ON oi.product_id = p.product_id
LEFT JOIN product_category_name t 
    ON p.product_category_name = t.product_category_name
WHERE t.product_category_name_english IS NOT NULL
ORDER BY o.order_id
INTO OUTFILE 'C:/ProgramData/MySQL/MySQL Server 8.0/Uploads/olist_seasonal_association.csv'
FIELDS TERMINATED BY ','
ENCLOSED BY '"'
LINES TERMINATED BY '\n';
'''

'\nHere is the sql query for extracting useful features.\nSELECT \n    o.order_id,\n    t.product_category_name_english,\n    o.order_purchase_timestamp,\n    MONTH(o.order_purchase_timestamp) AS month,\n    CASE \n        WHEN MONTH(o.order_purchase_timestamp) IN (12, 1, 2)  THEN \'Winter\'\n        WHEN MONTH(o.order_purchase_timestamp) IN (3, 4, 5)   THEN \'Spring\'\n        WHEN MONTH(o.order_purchase_timestamp) IN (6, 7, 8)   THEN \'Summer\'\n        WHEN MONTH(o.order_purchase_timestamp) IN (9, 10, 11) THEN \'Fall\'\n    END AS season\nFROM order_items oi\nJOIN orders o ON oi.order_id = o.order_id\nJOIN products p ON oi.product_id = p.product_id\nLEFT JOIN product_category_name t \n    ON p.product_category_name = t.product_category_name\nWHERE t.product_category_name_english IS NOT NULL\nORDER BY o.order_id\nINTO OUTFILE \'C:/ProgramData/MySQL/MySQL Server 8.0/Uploads/olist_seasonal_association.csv\'\nFIELDS TERMINATED BY \',\'\nENCLOSED BY \'"\'\nLINES TERMINATED BY \'\n\';\n'

In [27]:
orders = pd.read_csv('Seasonal_Association.csv',header=None,names=['order_id', 'category', 'timestamp', 'month', 'season'])
orders

,order_id,category,timestamp,month,season
0,00010242fe8c5a6d1ba2dd792cb16214,cool_stuff,2017-09-13 08:59:02,9,Fall
1,00018f77f2f0320c557190d7a144bdd3,pet_shop,2017-04-26 10:53:06,4,Spring
2,000229ec398224ef6ca0657da4fc703e,furniture_decor,2018-01-14 14:33:31,1,Winter
3,00024acbcdf0a6daa1e931b038114c75,perfumery,2018-08-08 10:00:35,8,Summer
4,00042b26cf59d7ce69dfabb4e55b4fd9,garden_tools,2017-02-04 13:57:51,2,Winter
...,...,...,...,...,...
111041,fffc94f6ce00a00581880bf54a75a037,housewares,2018-04-23 13:57:06,4,Spring
111042,fffcd46ef2263f404302a634eb57f7eb,computers_accessories,2018-07-14 10:26:46,7,Summer
111043,fffce4705a9662cd70adb13d4a31832d,sports_leisure,2017-10-23 17:07:56,10,Fall
111044,fffe18544ffabc95dfada21779c9644f,computers_accessories,2017-08-14 23:02:59,8,Summer


In [28]:
def run_seasonal_rules(season_df, season_name, min_support=0.01, min_lift=1.5):
    basket = season_df.groupby(['order_id', 'category'])['category'] \
                      .count().unstack(fill_value=0)
    basket = basket.map(lambda x: True if x > 0 else False)

    n = len(basket)
    print(f"\n{season_name}: {n} orders, {basket.shape[1]} categories")

    frequent_items = fpgrowth(basket, min_support=min_support, use_colnames=True)
    if frequent_items.empty:
        print(f"  → No frequent itemsets found")
        return pd.DataFrame()

    rules = association_rules(frequent_items, metric='lift', min_threshold=min_lift)
    if rules.empty:
        print(f"  → No rules found")
        return pd.DataFrame()

    print(f"  → {len(rules)} candidate rules found")

    results = []
    for _, row in rules.iterrows():
        ant = list(row['antecedents'])[0]
        con = list(row['consequents'])[0]

        count_both = int(row['support'] * n)
        count_ant  = int(basket[ant].sum()) if ant in basket.columns else 0
        base_rate  = basket[con].sum() / n  if con in basket.columns else 0

        if count_ant == 0 or base_rate == 0:
            continue

        stat, p_value = proportions_ztest(count_both, count_ant, base_rate)

        results.append({
            'season':      season_name,
            'rule':        f"{ant} → {con}",
            'antecedent':  ant,
            'consequent':  con,
            'support':     round(row['support'], 4),
            'confidence':  round(row['confidence'], 4),
            'lift':        round(row['lift'], 4),
            'z_stat':      round(stat, 4),
            'p_value':     round(p_value, 6),
            'significant': p_value < 0.05
        })

    result_df = pd.DataFrame(results)
    sig_count = result_df['significant'].sum()
    print(f"  → {sig_count} significant rules (p < 0.05)")
    return result_df

In [29]:
all_results = []

for season in ['Summer', 'Fall', 'Winter', 'Spring']:
    season_data = orders[orders['season'] == season]
    result = run_seasonal_rules(season_data, season, 
                                min_support=0.005,  
                                min_lift=1.5)
    if not result.empty:
        all_results.append(result)

if all_results:
    seasonal_df = pd.concat(all_results, ignore_index=True)
    print(seasonal_df[seasonal_df['significant']][['season','rule','lift','p_value']])
else:
    print("Still empty — paste Step 2 output for diagnosis")


Summer: 30106 orders, 72 categories
  → No rules found

Fall: 16351 orders, 70 categories
  → No rules found

Winter: 21573 orders, 70 categories
  → No rules found

Spring: 29246 orders, 72 categories
  → No rules found
Still empty — paste Step 2 output for diagnosis


In [30]:
all_results = []

for season in ['Summer', 'Fall', 'Winter', 'Spring']:
    season_data = orders[orders['season'] == season]
    result = run_seasonal_rules(season_data, season,
                                min_support=0.001,  
                                min_lift=1.2)      

    if not result.empty:
        all_results.append(result)

if all_results:
    seasonal_df = pd.concat(all_results, ignore_index=True)
    sig = seasonal_df[seasonal_df['significant'] == True]
    print(f"\nTotal significant rules: {len(sig)}")
    print(sig[['season','rule','support','confidence','lift','p_value']].to_string())
else:
    print("Still empty — try min_support=0.005")


Summer: 30106 orders, 72 categories
  → No rules found

Fall: 16351 orders, 70 categories
  → No rules found

Winter: 21573 orders, 70 categories
  → No rules found

Spring: 29246 orders, 72 categories
  → No rules found
Still empty — try min_support=0.005


In [31]:
for season in ['Summer', 'Fall', 'Winter', 'Spring']:
    season_data = orders[orders['season'] == season]
    items_per_order = season_data.groupby('order_id')['category'].count()
    multi = (items_per_order > 1).sum()
    total = len(items_per_order)
    print(f"{season}: {multi}/{total} multi-item orders ({round(multi/total*100,1)}%)")

Summer: 2858/30106 multi-item orders (9.5%)
Fall: 1740/16351 multi-item orders (10.6%)
Winter: 2087/21573 multi-item orders (9.7%)
Spring: 2970/29246 multi-item orders (10.2%)
